In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "Scanning/bin/Debug/net9.0/Scanning.dll"
open System
open System.IO
open LectureScanning
// Run python FSharp/tools/build.py from the repository root first.

# Converting <span style="font-variant:small-caps;">Html</span> to Text

This notebook shows how we can use the package [`FsLexYacc`](https://fsprojects.github.io/FsLexYacc/)
to extract the text that is embedded in an <span style="font-variant:small-caps;">Html</span> file.  
In order to be concise, it only supports a small subset of 
<span style="font-variant:small-caps;">Html</span>.  Below is the content of my old
<a href="http://wwwlehre.dhbw-stuttgart.de/~stroetma/">web page</a> that I had used when I was still working at the DHBW Stuttgart.  The goal of this notebook is to write 
a scanner that is able to extract the text from this web page.

In [ ]:
let data = """
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Homepage of Prof. Dr. Karl Stroetmann</title>
    <link type="text/css" rel="stylesheet" href="style.css" />
    <link href="http://fonts.googleapis.com/css?family=Rochester&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Pacifico&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Cabin+Sketch&subset=latin,latin-ext" rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Sacramento" rel="stylesheet" type="text/css">
  </head>
  <body>
    <hr/>

    <div id="table">
      <header>
        <h1 id="name">Prof. Dr. Karl Stroetmann</h1>
      </header>

      <div id="row1">
        <div class="right">
          <a id="dhbw" href="http://www.ba-stuttgart.de">Duale Hochschule Baden-W&uuml;rttemberg</a>
          <br/>Coblitzallee 1-9
          <br/>68163 Mannheim
          <br/>Germany
	  <br>
          <br/>Office: &nbsp;&nbsp;&nbsp; Raum 344B
          <br/>Phone:&nbsp;&nbsp;&nbsp; +49 621 4105-1376
          <br/>Fax:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; +49 621 4105-1194
          <br/>Skype: &nbsp;&nbsp;&nbsp; karlstroetmann
        </div>  


        <div id="links">
          <strong class="some">Some links:</strong>
          <ul class="inlink">
            <li class="inlink">
	      My <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">lecture notes</a>,
              as well as the programs presented in class, can be found
              at <br>
              <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">https://github.com/karlstroetmann</a>.
              
            </li>
            <li class="inlink">Most of my papers can be found at <a class="inlink" href="https://www.researchgate.net/">researchgate.net</a>.</li>
            <li class="inlink">The programming language SetlX can be downloaded at <br>
              <a href="http://randoom.org/Software/SetlX"><tt class="inlink">http://randoom.org/Software/SetlX</tt></a>.
            </li>
          </ul>
        </div>
      </div>
    </div>
    
    <div id="intro">
      As I am getting old and wise, I have to accept the limits of
      my own capabilities.  I have condensed these deep philosophical
      insights into a most beautiful pearl of poetry.  I would like 
      to share these humble words of wisdom:
      
      <div class="poetry">
        I am a teacher by profession,    <br>
        mostly really by obsession;      <br>
        But even though I boldly try,    <br>
        I just cannot teach <a href="flying-pig.jpg" id="fp">pigs</a> to fly.</br>
        Instead, I slaughter them and fry.
      </div>
      
      <div class="citation">
        <div class="quote">
          Any sufficiently advanced poetry is indistinguishable from divine wisdom.
        </div>
        <div id="sign">His holiness Pope Hugo &#8555;.</div>
      </div>
    </div>
</div>

</body>
</html>
"""

In [ ]:
printfn "%s" data

Compare the original page with the stripped text below. The source page and its linked illustration are local course material.

![The page links to this flying-pig illustration.](flying-pig.jpg)


In [ ]:
let preview = data.Replace("flying-pig.jpg", "data:image/jpeg;base64," + Convert.ToBase64String(File.ReadAllBytes "flying-pig.jpg"))
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(preview), "text/html") |> ignore


The original web page is still available at https://wwwlehre.dhbw-stuttgart.de/~stroetma/.

## Imports

FsLex builds a stateful scanner from `Scanning/Html.fsl`. The default state emits text; mutually recursive rules discard the contents of `head` and `script` elements. This is a teaching scanner for the shown HTML subset, rather than a general HTML DOM parser.

## Token Declarations

This scanner produces `string option`: `Some text` emits a piece of body text and `None` signals the end. Rules that discard tags or hidden sections recursively request the next emitted piece.

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
File.ReadAllText("Scanning/Html.fsl") |> printfn "%s"

## Definition of the States

Once we are inside an <span style="font-variant:small-caps;">Html</span> header or inside of some
*JavaScript* code the rules of the scanning game change.  Therefore, we declare two new <em style="color:blue">exclusive scanner states</em>:
- `header` is the state the scanner is in while it is scanning an 
  <span style="font-variant:small-caps;">Html</span> header.
- `script` is the state of the scanner while scanning *JavaScript* code.  

These states are *exclusive* states and hence the other token definitions do not apply in these
states.

In [ ]:
// States are recursive FsLex rules: token, header and script.

## Token Definitions

We proceed to give the definition of the tokens.  Note that none of the function defined below
returns a token.  Rather all of these function print the transformation of the 
<span style="font-variant:small-caps;">Html</span> that they have matched.

### The Definition of the Token `HEAD_START`

Matching `<head>` calls the recursive `header` rule. This rule discards characters until the closing head tag, so page metadata does not become body text.

Once the scanner reads the opening tag `<head>` it switches into the state `header`.  The function `begin` of the lexer can be used to switch into a different scanner state.  In the state `header`, the scanner continues to read and discard characters until the closing tag `</head>` is encountered.  Note that this token is only recognized in the state `INITIAL`.  The state `INITIAL` is the initial state of the scanner, i.e. the scanner always starts in this state.

In [ ]:
let rule19 = """| "<head>" { header lexbuf }"""
printfn "%s" rule19

### The Definition of the Token `SCRIPT_START`

The opening script tag calls the recursive `script` rule; attributes are allowed. Script contents are discarded until `</script>`.

In [ ]:
let rule21 = """| "<script" [^ '>']* '>' { script lexbuf }"""
printfn "%s" rule21

### The Definition of the Token `LINEBREAK``

A newline and surrounding whitespace produce one newline string. Consecutive empty lines therefore collapse into a single line break.

In [ ]:
let rule23 = """| [' ' '\t' '\r']* '\n' [' ' '\t' '\r' '\n']* { Some "\n" }"""
printfn "%s" rule23

### The Definition of the Token `TAG`

An opening angle bracket, characters through the next closing bracket, and that closing bracket are discarded. This deliberately models the small HTML subset of the exercise.

In [ ]:
let rule25 = """| '<' [^ '>']* '>' { token lexbuf }"""
printfn "%s" rule25

### The Definition of the Token `NAMED_ENTITY`

Named entities are decoded with .NET `WebUtility.HtmlDecode`. The lexer accepts an optional final semicolon; unknown entity names remain unchanged.

In [ ]:
open System.Net

In [ ]:
WebUtility.HtmlDecode("&auml;")

.NET `WebUtility.HtmlDecode` decodes named entities. The generated scanner emits the decoded string, leaving unknown entities unchanged.

In [ ]:
let rule30 = """| '&' ['a'-'z' 'A'-'Z']+ ';'? { Some(WebUtility.HtmlDecode(text lexbuf)) }"""
printfn "%s" rule30

### Unicode entities

Decimal and hexadecimal entity patterns feed .NET HTML decoding, supporting Unicode scalar values including characters outside the BMP.

In [ ]:
let rule32 = """| "&#" ['0'-'9']+ ';'? { Some(WebUtility.HtmlDecode(text lexbuf)) }"""
printfn "%s" rule32

In [ ]:
Char.ConvertFromUtf32(8555)

In [ ]:
Char.ConvertFromUtf32(128034)

### The Definition of the Token `ANY` 

The `_` pattern matches any remaining character, including a newline. In the default rule it emits text; in `header` and `script` it discards the character and recurses.

In [ ]:
let rule36 = """| _ { Some(text lexbuf) }"""
printfn "%s" rule36

### The Definition of the Token `HEAD_END` 

The `header` rule recognizes `</head>` and returns to the default `token` rule. No mutable string state or dynamically added lexer attribute is required.

In [ ]:
let rule38 = """| "</head>" { token lexbuf }"""
printfn "%s" rule38

### The Definition of the Token `SCRIPT_END`

The `script` rule recognizes `</script>` and returns to the default rule.

The regular expression `</script>` matches the closing script tag.  Note that this regular expression is only
active in state `script`.  Once the closing tag has been found, the function `lexer.begin` switches the lexer back into the state `INITIAL`, which is the start state of the scanner.  

In [ ]:
let rule41 = """| "</script>" { token lexbuf }"""
printfn "%s" rule41

### The Definition of the Token `ANY`

The `_` pattern matches any remaining character, including a newline. In the default rule it emits text; in `header` and `script` it discards the character and recurses.

In [ ]:
let rule43 = """| "<head>" { header lexbuf }
| _ { header lexbuf }"""
printfn "%s" rule43

## Error Handling

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let rule46 = """| eof { None }
| eof { failwith "Unclosed HTML head" }
| eof { failwith "Unclosed HTML script" }"""
printfn "%s" rule46

An end-of-input token while discarding a head or script section reports an unclosed section. All other characters match `_`, so no character can get stuck in these states.

In [ ]:
let rule48 = """| eof { failwith "Unclosed HTML head" }"""
printfn "%s" rule48

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let rule50 = """| eof { failwith "Unclosed HTML script" }"""
printfn "%s" rule50

## Running the Scanner

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let convertHtml = Scanning.html

Next, we feed our input string into the generated scanner.

In [ ]:
let source = data

In order to scan the data that we provided in the last line, we iterate over all tokens generated by our scanner.

In [ ]:
let scan source = convertHtml source

In [ ]:
let plainText = scan source
printfn "%s" plainText